# Harmful Text Detection — Build Augmented Dataset

Produces `toxicity_augmented.csv` (~30 K rows) for upload to Kaggle.  
`TRAIN_MODEL.ipynb` auto-detects this file in `/kaggle/input/`.

## Label schema — 5 labels
| Label | Notes |
|-------|-------|
| `toxicity` | General toxic/rude language. **Absorbs `severe_toxicity`** — the severity distinction was noisy and poorly annotated. |
| `obscene` | Profane / explicit language |
| `threat` | Threats of harm, reputational damage, doxxing |
| `insult` | Direct insults and demeaning statements |
| `identity_hate` | Hate speech targeting race, nationality, ethnicity, religion, etc. |

## Dataset ratio
- **50 % toxic / 50 % clean** — N_RARE=8 000 (identity_hate + threat), N_OTHER=7 000, N_CLEAN=15 000
- Civil Comments sampled identity-hate-first for better coverage
- Synthetic examples upsampled 30× and cover all 4 platform surfaces

## Sources
| Source | Rows | Strength |
|--------|------|----------|
| Jigsaw (HF) | ~15 K | Broad toxicity baseline |
| Civil Comments (HF) | ~12 K | Real identity hate, nuanced bias |
| Synthetic professional | ~3 K (120×30) | Platform-specific patterns the model misses |

In [ ]:
!pip install -q datasets pandas numpy
print('Done')

In [ ]:
import os, random
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# 5-label schema — severe_toxicity merged into toxicity
LABELS      = ['toxicity', 'obscene', 'threat', 'insult', 'identity_hate']
RARE_LABELS = ['identity_hate', 'threat']
SCORE_THR   = 0.5
SYNTH_REPS  = 30
OUTPUT      = 'toxicity_augmented.csv'

# Target sizes → 50/50 split
N_RARE  = 8000   # rare-label rows (identity_hate, threat)
N_OTHER = 7000   # other toxic (toxicity / obscene / insult only)
N_CLEAN = 15000  # clean rows

print(f'Labels : {LABELS}')
print(f'Target : {N_RARE+N_OTHER} toxic + {N_CLEAN} clean = {N_RARE+N_OTHER+N_CLEAN:,} rows')

## 1. Jigsaw Toxic Comment Classification

Same source as `TRAIN_MODEL.ipynb`.  
`severe_toxic` → merged into `toxicity` via OR.

In [ ]:
print('Loading Jigsaw...')
jigsaw_raw = load_dataset('thesofakillers/jigsaw-toxic-comment-classification-challenge', split='train')
jigsaw_df  = jigsaw_raw.to_pandas()
jigsaw_df  = jigsaw_df.rename(columns={'comment_text': 'text', 'toxic': 'toxicity', 'severe_toxic': 'severe_toxicity'})

# Merge severe_toxicity → toxicity
jigsaw_df['toxicity'] = ((jigsaw_df['toxicity'] == 1) | (jigsaw_df.get('severe_toxicity', 0) == 1)).astype(int)

for lbl in LABELS:
    jigsaw_df[lbl] = jigsaw_df[lbl].fillna(0).astype(int) if lbl in jigsaw_df.columns else 0

jigsaw_df = jigsaw_df[['text'] + LABELS].dropna(subset=['text'])
jigsaw_df['text']   = jigsaw_df['text'].astype(str).str.strip()
jigsaw_df           = jigsaw_df[jigsaw_df['text'].str.len() > 10].reset_index(drop=True)
jigsaw_df['source'] = 'jigsaw'

any_toxic = jigsaw_df[LABELS].sum(axis=1) > 0
print(f'Rows: {len(jigsaw_df):,}  |  Toxic: {any_toxic.sum():,} ({any_toxic.mean()*100:.1f}%)')
for lbl in LABELS:
    n = jigsaw_df[lbl].sum()
    print(f'  {lbl:20s}: {n:6,} ({n/len(jigsaw_df)*100:.2f}%)')

## 2. Civil Comments (Jigsaw Unintended Bias)

`identity_attack` → `identity_hate`.  
`severe_toxicity` → merged into `toxicity`.  
Sampled identity-hate-first.

In [ ]:
print('Loading Civil Comments (may take a few minutes)...')
try:
    civil_raw = load_dataset('civil_comments', split='train')
except Exception:
    civil_raw = load_dataset('google/civil_comments', split='train')

civil_df = civil_raw.to_pandas()
civil_df = civil_df.rename(columns={'identity_attack': 'identity_hate'})
text_col = 'text' if 'text' in civil_df.columns else 'comment_text'
civil_df = civil_df.rename(columns={text_col: 'text'})

# Merge severe_toxicity → toxicity
if 'severe_toxicity' in civil_df.columns:
    civil_df['toxicity'] = np.maximum(civil_df.get('toxicity', 0), civil_df['severe_toxicity'])

for lbl in LABELS:
    if lbl in civil_df.columns:
        civil_df[lbl] = (civil_df[lbl] >= SCORE_THR).astype(int)
    else:
        civil_df[lbl] = 0

civil_df = civil_df[['text'] + LABELS].dropna(subset=['text'])
civil_df['text']   = civil_df['text'].astype(str).str.strip()
civil_df           = civil_df[civil_df['text'].str.len() > 10].reset_index(drop=True)
civil_df['source'] = 'civil_comments'

id_rows    = civil_df[civil_df['identity_hate'] == 1]
other_tox  = civil_df[(civil_df[LABELS].sum(axis=1) > 0) & (civil_df['identity_hate'] == 0)]
clean_rows = civil_df[civil_df[LABELS].sum(axis=1) == 0]

civil_sampled = pd.concat([
    id_rows.sample(n=min(len(id_rows), 5000),    random_state=SEED),
    other_tox.sample(n=min(len(other_tox), 4000), random_state=SEED),
    clean_rows.sample(n=min(len(clean_rows), 3000), random_state=SEED),
]).drop_duplicates(subset=['text']).reset_index(drop=True)

print(f'Civil sample: {len(civil_sampled):,}  |  identity_hate: {civil_sampled["identity_hate"].sum():,}')

## 3. Synthetic Professional Examples

**Curation rule:** added only if unambiguously toxic with zero legitimate professional interpretation.

**Removed (not included):**
- Language-only requirements (`native English speakers only`) — legitimate
- Generic local preference (`hiring domestically`, `local candidates preferred`) — legitimate
- Blunt-but-professional feedback (`brief is vague`, `skills don't match portfolio`) — not hate/threat

**Covers all 4 platform surfaces:** job post · profile · proposal · message  
**Clean examples include:** contract termination, proposal rejection, work feedback — text the model must NOT flag.

In [ ]:
# Tuple format: (text, toxicity, obscene, threat, insult, identity_hate)
# severe_toxicity is gone — merged into toxicity
synthetic_raw = [

    # ── IDENTITY HATE — "your type / kind / sort / people" coded language ──────
    ('People like you from that country always try to underbid then disappear with the money', 1,0,0,1,1),
    ('I do not work with your kind, you always cause delays and make excuses', 1,0,0,1,1),
    ('Your type of freelancers never delivers what they promise, always cutting corners', 1,0,0,1,1),
    ('We have had enough of people like you who do not respect professional standards', 1,0,0,1,1),
    ('Someone from your background obviously would not understand what quality looks like', 1,0,0,1,1),
    ('Your kind always tries to renegotiate the price after the work is done', 1,0,0,0,1),
    ('People from your country are known for scamming clients, I should have known better', 1,0,0,1,1),
    ('I would not trust your sort with anything important, you are all the same', 1,0,0,1,1),
    ('Freelancers like you from developing countries think they can charge premium rates for subpar work', 1,0,0,1,1),
    ('Your people have a different definition of done that does not match professional standards', 1,0,0,1,1),
    ('I can always tell when I am dealing with someone from your region, it never ends well', 1,0,0,0,1),
    ('Your type never takes feedback well, always defensive and making excuses', 1,0,0,1,1),
    ('Where you are from, maybe this passes as acceptable work, but not here', 1,0,0,1,1),
    ('People of your background always overpromise and underdeliver, no exception here', 1,0,0,0,1),
    ('I have worked with your type before and you people always find a way to disappoint', 1,0,0,1,1),
    ('I see from your name where you are from and I have had nothing but trouble with your sort', 1,0,0,1,1),
    ('Developers from your region have a reputation for this and you are proving it right', 1,0,0,1,1),
    ('Your kind of people never understand the value of professional commitment', 1,0,0,0,1),
    ('I am not paying someone like you, your type always scams people and then disappears', 1,0,0,1,1),
    ('Your sort never delivers quality and you are no exception to that rule', 1,0,0,1,1),
    ('People like you are why this platform has a reputation problem, go back to where you came from', 1,0,0,1,1),
    ('Your kind of contractor is exactly what gives this industry a bad name', 1,0,0,1,1),
    ('Do not hire people from that country, they are all the same and you will regret it', 1,0,0,0,1),
    ('Do not bother applying if you are from certain countries, we know how that ends', 1,0,0,0,1),
    ('I see from your profile where you are from and I have worked with your type before, you people always cause the same issues', 1,0,0,1,1),

    # ── DISCRIMINATORY HIRING — explicit group inferiority, not just local preference ──
    ('We do not hire freelancers from South Asia or Southeast Asia based on past experience', 1,0,0,0,1),
    ('We specifically avoid contractors from certain countries because the quality is always disappointing', 1,0,0,0,1),
    ('This role is only open to local candidates, we do not hire foreign workers because they always create problems', 1,0,0,0,1),
    ('Do not apply if you are from a country with known quality issues in this field', 1,0,0,0,1),
    ('Foreign freelancers from certain regions have consistently failed to deliver and we no longer consider them', 1,0,0,0,1),

    # ── VEILED THREATS ──────────────────────────────────────────────────────────
    ('Meet every deadline without exception, miss one and I will make sure your reputation on this platform is destroyed', 1,0,1,0,0),
    ('Pay me the full amount immediately or I will report you on every platform and post your information publicly', 1,0,1,0,0),
    ('If you dispute this I will contact all your other clients and let them know what you really are', 1,0,1,0,0),
    ('You have 24 hours to fix this or I will go to every freelance forum and expose your incompetence', 1,0,1,0,0),
    ('The last freelancer who tried to cross me lost every major client within a month, think about that carefully', 1,0,1,0,0),
    ('Reject my proposal and I guarantee you will regret it professionally, I have connections in this industry', 1,0,1,0,0),
    ('Give me a full refund now or I will make it my mission to bury your ratings on every review site', 1,0,1,0,0),
    ('You would be making a serious mistake not hiring me, the last client who rejected me found that out', 1,0,1,0,0),
    ('Do what I say or I will use every channel available to damage your professional reputation permanently', 1,0,1,0,0),
    ('Correct this immediately or there will be serious professional consequences for you personally', 1,0,1,0,0),
    ('I will ruin you and your business if this is not resolved by end of day today', 1,0,1,0,0),
    ('Missing this deadline will be the worst career decision you have ever made, I promise you that', 1,0,1,0,0),
    ('You will deliver exactly what I asked for or I will take legal action and make sure it costs you everything', 1,0,1,0,0),
    ('Pay me the full amount by tomorrow or I will report you on every platform I can find and make sure everyone knows', 1,0,1,0,0),
    ('I know exactly who you are and where your business operates, do not make me escalate this further', 1,0,1,0,0),
    ('I will find you and make you pay for what you did with my money, this is not over', 1,0,1,0,0),
    ('You stole from me and I will destroy you professionally and personally for this', 1,0,1,0,0),
    ('I will hunt you down if you do not refund me immediately, do not test me on this', 1,0,1,0,0),

    # ── PROFESSIONAL INSULTS — personal, not just harsh feedback ────────────────
    ('Honestly your work is an embarrassment for someone charging these rates, completely amateur hour', 1,0,0,1,0),
    ('I do not know how you have any positive reviews, this is objectively terrible work', 1,0,0,1,0),
    ('Any competent developer would have done this correctly the first time, what is wrong with you', 1,0,0,1,0),
    ('You call yourself a professional? This belongs in a beginner tutorial, not a paid project', 1,0,0,1,0),
    ('I have worked with hundreds of freelancers and you are by far the most incompetent one I have ever encountered', 1,0,0,1,0),
    ('Only a complete idiot would submit work like this and expect to be paid for it', 1,0,0,1,0),
    ('Seriously how are you even a developer, this is not even close to what I described', 1,0,0,1,0),

    # ── OBSCENE ─────────────────────────────────────────────────────────────────
    ('This is absolute garbage work, what the hell were you thinking when you submitted this crap', 1,1,0,1,0),
    ('Go to hell you useless fraud, take your shit code and shove it', 1,1,0,1,0),
    ('You are an absolute piece of trash, this work is complete bullshit and you know it', 1,1,0,1,0),
    ('What the fuck is this garbage, did a child write this code', 1,1,0,1,0),
    ('You are a pathetic excuse for a developer, this shit is embarrassing to look at', 1,1,0,1,0),

    # ── PLATFORM-SURFACE TOXIC — job post ───────────────────────────────────────
    ('If the freelancer we hire fails to deliver on time we will report them on every platform and ensure they never work again', 1,0,1,0,0),
    ('We have been burned by overseas contractors too many times, your kind of people is exactly what we are trying to avoid', 1,0,0,1,1),
    ('Only developers from first world countries need apply, freelancers from third world countries have consistently wasted our time', 1,0,0,0,1),

    # ── PLATFORM-SURFACE TOXIC — profile (freelancer bio) ───────────────────────
    ('I will destroy any client who tries to underpay me, the last one who did regretted it, do not test me', 1,0,1,0,0),
    ('Cheap clients who lowball freelancers are the absolute scum of this platform, save us both time and go elsewhere', 1,0,0,1,0),
    ('If you are thinking of disputing my payment after I deliver, I already know how to make your life on this platform very difficult', 1,0,1,0,0),

    # ── PLATFORM-SURFACE TOXIC — proposal ───────────────────────────────────────
    ('Your project brief is so poorly written it is clear you have absolutely no idea what you actually need', 1,0,0,1,0),
    ('Any developer worth their rate would tell you this brief is unacceptably vague, do you even know what you want to build', 1,0,0,1,0),
    ('If you do not select me for this project I can assure you that your chosen developer will fail and you will come crawling back', 1,0,1,0,0),

    # ── PLATFORM-SURFACE TOXIC — messages ───────────────────────────────────────
    ('I know where your company is registered and I have already contacted the authorities, pay me now or this escalates further', 1,0,1,0,0),
    ('Your kind of client always tries to avoid payment, I have dealt with people from your country before and it always ends this way', 1,0,0,1,1),
    ('Stop ignoring my messages or I will post your personal information and company details on every forum I can find', 1,0,1,0,0),
    ('You are the worst client I have ever dealt with on any platform, an absolute disgrace of a human being', 1,0,0,1,0),
    ('If you leave me a negative review I will report your account and dispute every transaction you have ever made on this platform', 1,0,1,0,0),

    # ── CLEAN — contract termination and rejection (must NOT be flagged) ─────────
    ('After reviewing your deliverables we have decided to close this contract as the work does not meet the agreed requirements', 0,0,0,0,0),
    ('We appreciate your effort but the project is not heading in the right direction, we will be ending the engagement here', 0,0,0,0,0),
    ('We are terminating this contract due to consistent missed deadlines and misalignment with the agreed scope', 0,0,0,0,0),
    ('This is not the quality we expected and we will not be proceeding further, payment for completed milestones will be sent as agreed', 0,0,0,0,0),
    ('After careful evaluation we have chosen to work with a different developer, thank you for your time', 0,0,0,0,0),
    ('Your proposal was not selected for this project, we wish you success in future opportunities', 0,0,0,0,0),
    ('We are pausing this project due to budget constraints and will not need further work at this time', 0,0,0,0,0),
    ('The deliverable does not meet our specifications, we are requesting a revision before the final payment is released', 0,0,0,0,0),
    ('I want to end our working relationship as the project is not meeting expectations, payment for completed work will be approved', 0,0,0,0,0),
    ('We need to formally close this contract, your final invoice for completed milestones has been approved for payment', 0,0,0,0,0),
    ('After reviewing all proposals we have selected another candidate, we appreciate the time you put into your application', 0,0,0,0,0),
    ('We have decided to go in a different direction and will not be renewing this contract, thank you for your contributions', 0,0,0,0,0),

    # ── CLEAN — job posts ────────────────────────────────────────────────────────
    ('We are looking for a freelance content writer with experience in SaaS products, 10 to 15 articles per month', 0,0,0,0,0),
    ('Seeking a mobile developer for iOS app development, minimum 3 years of Swift experience required', 0,0,0,0,0),
    ('We are a startup building a marketplace app and need a React developer for a 6 week project', 0,0,0,0,0),
    ('Looking for an experienced UI UX designer for a product redesign, Figma proficiency required', 0,0,0,0,0),
    ('Need a data analyst to help us build reporting dashboards, experience with Tableau or Power BI preferred', 0,0,0,0,0),

    # ── CLEAN — profile bios ─────────────────────────────────────────────────────
    ('I am a freelance graphic designer specializing in brand identity and logo design for early stage startups', 0,0,0,0,0),
    ('Full stack developer with 7 years of experience in Python and Django, available for long term remote contracts', 0,0,0,0,0),
    ('Marketing specialist focused on email campaigns and SEO, 100 percent on time delivery across 80 plus projects', 0,0,0,0,0),
    ('I build clean and accessible web applications, my clients range from solo founders to mid size companies', 0,0,0,0,0),
    ('Certified project manager with experience running agile teams, specializing in software delivery and client communication', 0,0,0,0,0),

    # ── CLEAN — proposals ────────────────────────────────────────────────────────
    ('I have reviewed your project brief and can deliver the MVP within 4 weeks, here is my proposed timeline and milestones', 0,0,0,0,0),
    ('Based on my experience with similar e-commerce platforms I suggest the following architecture for your project', 0,0,0,0,0),
    ('I am confident I can meet your requirements within budget, I have attached relevant portfolio samples for your review', 0,0,0,0,0),
    ('I noticed a few gaps in the brief that I would like to clarify before starting, could we schedule a quick call', 0,0,0,0,0),
    ('My approach would be to start with a discovery phase before committing to a full scope, this reduces risk on both sides', 0,0,0,0,0),

    # ── CLEAN — messages ─────────────────────────────────────────────────────────
    ('Hi just following up on the UI mockups, could you confirm if the color palette we discussed is approved', 0,0,0,0,0),
    ('The first phase is complete, please review the staging environment and let me know if there are any changes needed', 0,0,0,0,0),
    ('I need a few more days to complete the integration, the API documentation was more complex than expected', 0,0,0,0,0),
    ('Payment for milestone 2 has been released, looking forward to starting phase 3 next week', 0,0,0,0,0),
    ('Could you clarify the expected behavior when a user submits the form with missing fields', 0,0,0,0,0),
    ('I have pushed the latest changes to the repository, the build is passing and ready for your review', 0,0,0,0,0),
    ('Just checking in on the status of the review, we have a client demo scheduled for Friday', 0,0,0,0,0),
    ('We are looking for an experienced Python developer for a 3-month remote contract', 0,0,0,0,0),
    ('This does not meet the requirements outlined in the brief, please review section 3 and resubmit', 0,0,0,0,0),
    ('The code quality is good but we need better documentation before we can sign off on this milestone', 0,0,0,0,0),
]

synth_df = pd.DataFrame(synthetic_raw, columns=['text'] + LABELS)
synth_df['source'] = 'synthetic'

toxic = (synth_df[LABELS].sum(axis=1) > 0).sum()
clean = len(synth_df) - toxic
print(f'Synthetic base: {len(synth_df)} ({toxic} toxic, {clean} clean)')
print('Per-label (base):')
for lbl in LABELS:
    n = synth_df[lbl].sum()
    if n: print(f'  {lbl:20s}: {n}')

## 4. Combine → Sample to ~30 K → Save

In [ ]:
synth_up = pd.concat([synth_df] * SYNTH_REPS, ignore_index=True)

combined = pd.concat([jigsaw_df, civil_sampled, synth_up], ignore_index=True)
combined = combined[['text'] + LABELS + ['source']]
combined = combined.dropna(subset=['text'])
combined['text'] = combined['text'].astype(str).str.strip()
combined = combined[combined['text'].str.len() > 10]
combined = combined.drop_duplicates(subset=['text']).reset_index(drop=True)
for lbl in LABELS:
    combined[lbl] = combined[lbl].astype(int)

print(f'Combined (pre-sample): {len(combined):,}')

# Stratified sample to ~30 K with 50/50 balance
rare_mask  = combined[RARE_LABELS].sum(axis=1) > 0
rare_df    = combined[rare_mask]
other_tox  = combined[(combined[LABELS].sum(axis=1) > 0) & ~rare_mask]
clean_df   = combined[combined[LABELS].sum(axis=1) == 0]

print(f'  Rare-label rows  : {len(rare_df):,}')
print(f'  Other toxic rows : {len(other_tox):,}')
print(f'  Clean rows       : {len(clean_df):,}')

final = pd.concat([
    rare_df.sample(n=min(len(rare_df),  N_RARE),  random_state=SEED),
    other_tox.sample(n=min(len(other_tox), N_OTHER), random_state=SEED),
    clean_df.sample(n=min(len(clean_df),  N_CLEAN), random_state=SEED),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

any_toxic = final[LABELS].sum(axis=1) > 0
print(f'\nFinal: {len(final):,} rows')
print(f'  Toxic : {any_toxic.sum():,} ({any_toxic.mean()*100:.1f}%)')
print(f'  Clean : {(~any_toxic).sum():,} ({(~any_toxic).mean()*100:.1f}%)')
print(f'\nSource breakdown:')
print(final['source'].value_counts().to_string())
print(f'\nPer-label positives:')
for lbl in LABELS:
    n = final[lbl].sum()
    print(f'  {lbl:20s}: {n:5,} ({n/len(final)*100:.2f}%)')

In [ ]:
final.to_csv(OUTPUT, index=False)
mb = os.path.getsize(OUTPUT) / 1024 / 1024
print(f'Saved  : {OUTPUT}')
print(f'Rows   : {len(final):,}')
print(f'Size   : {mb:.1f} MB')
print(f'Columns: {list(final.columns)}')
print()
print('Next steps:')
print('  1. Download toxicity_augmented.csv')
print('  2. Create a Kaggle dataset and upload the file')
print('  3. Add that dataset to TRAIN_MODEL.ipynb on Kaggle')
print('  4. Run TRAIN_MODEL.ipynb — cell 2 auto-detects /kaggle/input/.../toxicity_augmented.csv')